In [ ]:
%run universal-framework.ipynb

In [ ]:
def create_lag_dataset(
    y,
    n_lags=14
):

    X = []
    target = []

    for i in range(n_lags, len(y)):

        X.append(
            y[i-n_lags:i]
        )

        target.append(
            y[i]
        )

    return (
        np.array(X),
        np.array(target)
    )

In [ ]:
def recursive_forecast(
    model,
    history,
    horizon,
    n_lags
):

    history = list(history)

    forecasts = []

    for _ in range(horizon):

        x = np.array(
            history[-n_lags:]
        ).reshape(1, -1)

        pred = model.predict(x)[0]

        forecasts.append(pred)

        history.append(pred)

    return np.array(forecasts)

In [ ]:
def lightgbm_forecast(
    train,
    horizon,
    n_lags=14
):

    if len(train) < n_lags + horizon:

        return np.repeat(
            train[-1],
            horizon
        )

    X, y = create_lag_dataset(
        train,
        n_lags
    )

    model = LGBMRegressor(
        n_estimators=100,
        learning_rate=0.05,
        num_leaves=31,
        verbosity=-1,
        random_state=42
    )

    model.fit(X, y)

    return recursive_forecast(
        model,
        train,
        horizon,
        n_lags
    )

In [ ]:
def xgboost_forecast(
    train,
    horizon,
    n_lags=14
):

    if len(train) < n_lags + horizon:
        return np.repeat(
            train[-1],
            horizon
        )

    X, y = create_lag_dataset(
        train,
        n_lags
    )

    model = XGBRegressor(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.05,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X, y)

    pred = recursive_forecast(
        model=model,
        history=train,
        horizon=horizon,
        n_lags=n_lags
    )

    return pred

In [ ]:
def run_tree_model(
    model_name,
    forecast_function,
    output_file
):

    results = []

    for file in DATA_DIR.glob("*.csv"):

        dataset_name = file.stem

        print(
            f"\nRunning {model_name} "
            f"on {dataset_name}"
        )

        df = pd.read_csv(
            file,
            low_memory=False
        )

        for horizon, horizon_type in get_horizons_info(
            dataset_name
        ):

            result = benchmark_dataset(
                df=df,
                horizon=horizon,
                forecast_function=forecast_function,
                model_name=model_name
            )

            if result is not None:

                result["dataset"] = dataset_name
                result["horizon_type"] = horizon_type

                results.append(result)

                pd.DataFrame(results).to_csv(
                    output_file,
                    index=False
                )

        del df
        gc.collect()

    return pd.DataFrame(results)

In [ ]:
lightgbm_results = run_tree_model(
    model_name="LightGBM",
    forecast_function=lightgbm_forecast,
    output_file=RESULTS_DIR / "lightgbm.csv"
)


Running LightGBM on std_agri_large

Running LightGBM on std_agri_medium

Running LightGBM on std_agri_small

Running LightGBM on std_energy_large

Running LightGBM on std_energy_medium

Running LightGBM on std_energy_small

Running LightGBM on std_favorita_large_train

Running LightGBM on std_favorita_large_transactions

Running LightGBM on std_favorita_medium_train

Running LightGBM on std_favorita_medium_transactions

Running LightGBM on std_favorita_small_train

Running LightGBM on std_favorita_small_transactions

Running LightGBM on std_m4_daily_large

Running LightGBM on std_m4_daily_medium

Running LightGBM on std_m4_daily_small

Running LightGBM on std_m4_hourly_large

Running LightGBM on std_m4_hourly_medium

Running LightGBM on std_m4_hourly_small

Running LightGBM on std_m4_monthly_large

Running LightGBM on std_m4_monthly_medium

Running LightGBM on std_m4_monthly_small

Running LightGBM on std_m4_quarterly_large

Running LightGBM on std_m4_quarterly_medium

Running LightGB

In [ ]:
xgboost_results = run_tree_model(
    model_name="XGBoost",
    forecast_function=xgboost_forecast,
    output_file=RESULTS_DIR / "xgboost.csv"
)


Running XGBoost on std_agri_large

Running XGBoost on std_agri_medium

Running XGBoost on std_agri_small

Running XGBoost on std_energy_large

Running XGBoost on std_energy_medium

Running XGBoost on std_energy_small

Running XGBoost on std_favorita_large_train

Running XGBoost on std_favorita_large_transactions

Running XGBoost on std_favorita_medium_train

Running XGBoost on std_favorita_medium_transactions

Running XGBoost on std_favorita_small_train

Running XGBoost on std_favorita_small_transactions

Running XGBoost on std_m4_daily_large

Running XGBoost on std_m4_daily_medium

Running XGBoost on std_m4_daily_small

Running XGBoost on std_m4_hourly_large

Running XGBoost on std_m4_hourly_medium

Running XGBoost on std_m4_hourly_small

Running XGBoost on std_m4_monthly_large


In [ ]:
display(
    pd.concat([
        lightgbm_results,
        xgboost_results
    ])
)